# Plug-in template · add your idea to the race

**Owner:** Tudor · **For:** Carla, Alex, Alexandra (and anyone with a new idea)

The race engine (`qgd_bench.ipynb`) has five **sockets**. Each one is a slot for one function, and you can swap that function for your own without touching the engine. This notebook shows one working example per socket and then runs a small race with all five plugged in.

**How to use it**

1. Copy this notebook to `bench/<your-name>.ipynb`.
2. Keep only the sockets you need. Rename each entry to start with your name, e.g. `@register("qaoa", "carla_ws")`.
3. Replace the body of each function with your idea, keeping the inputs and outputs described above it.
4. In the last section, pick your entries in `Config(...)` and run the race. The baseline runs next to you on the same puzzles.
5. Keep every race under `if not PLUGINS_ONLY:`. On merge day one notebook loads all four of ours with `PLUGINS_ONLY = True`, so only the `@register` cells run (section 8).

| Socket | Your function gets | It returns | Owner |
|---|---|---|---|
| `evidence` | reads | `(A, D)` vote weights | Alexandra |
| `qaoa` | `(J, p)` | `(circuit, parameters, scale)` | Carla |
| `trainer` | `(J, offset, p, circuit name, ctx)` | `(angles, training shots[, extra columns])` | Carla |
| `postprocess` | `(samples, J, offset)` | `(samples, extra cost evaluations)` | Alex |
| `backend` | `(job, shots, seed)` | `(samples, exact probabilities or None)` | Alex |

The fair-race rules **A–D** mentioned below are explained with pictures on page 3 (`docs/research/merge-plan.html`, section 4).

## 0 · Load the engine

`%run` executes `qgd_bench.ipynb` inside this notebook, so every engine function (`register`, `costs`, `run_bench`, …) is now defined here. `QGD_QUIET = True` skips the engine's own examples and checks.

`PLUGINS_ONLY` is `False` when you open this notebook yourself. The merge notebook sets it to `True` before loading yours, which skips your races and keeps only your plug-ins.

In [1]:
PLUGINS_ONLY = globals().get("PLUGINS_ONLY", False)
QGD_QUIET = True
%run qgd_bench.ipynb
import pandas as pd

print("sockets:", {kind: sorted(entries) for kind, entries in REGISTRY.items()})

sockets: {'evidence': ['pairwise'], 'qaoa': ['baseline'], 'trainer': ['baseline', 'cobyla'], 'postprocess': ['none'], 'backend': ['aer_mps', 'aer_sv', 'numpy']}


## 1 · Evidence socket: reads → votes

**Gets** `reads`: a matrix with one row per read and one column per SNP. Each value is `0`, `1`, or `-1` (no letter here).

**Returns** `(A, D)`: two N × N matrices. `A[i, j]` is how strongly the reads say SNPs i and j are on the **same** copy, and `D[i, j]` how strongly they say **different** copies. Only the upper triangle (`i < j`) is used.

The engine turns them into the cost with `J = (D − A) / 2` and `offset = Σ(A + D) / 2`. Weights do not have to be whole numbers: a read you trust less can vote 0.5.

⚠ **Rule C:** new weights change which answer scores best. Judge evidence ideas by `switch_err` (distance from the hidden truth), not by `P_budget`.

In [2]:
@register("evidence", "template_pairwise")
def template_evidence(reads):
    A, D = pairwise_evidence(reads)
    return A, D   # change the weights here (e.g. per-read quality); keep the upper-triangular shape

## 2 · Circuit socket: couplings → QAOA circuit

**Gets** `J` (the couplings) and the depth `p`.

**Returns** `(circuit, parameters, scale)`:
* `circuit`: a `QuantumCircuit` with N qubits and **no measurements**. Qubit i is SNP i + 1.
* `parameters`: a `ParameterVector`. The engine assigns trained angles to it in this order.
* `scale`: the number the couplings were divided by (`qaoa_scale(J)`, the largest |J|), so angles mean the same at every size.

The example keeps the baseline cost layer (`RZZ` on every linked pair) and swaps the `RX` mixer for `RY`.

Good to know: a new circuit runs on Aer automatically, because the fast numpy simulator only knows the baseline. With more than 2p parameters (e.g. multi-angle QAOA), also write a trainer that returns that many angles.

In [3]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector


@register("qaoa", "template_ry")
def template_circuit(J, p=1):
    n, scale = len(J), qaoa_scale(J)
    theta = ParameterVector("theta", 2 * p)          # [gamma_1 .. gamma_p, beta_1 .. beta_p]
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for layer in range(p):
        for a, b in zip(*np.nonzero(np.triu(J, 1))):
            qc.rzz(2 * theta[layer] * J[a, b] / scale, int(a), int(b))
        qc.ry(2 * theta[p + layer], range(n))         # baseline: qc.rx(...)
    return qc, theta, scale


if not PLUGINS_ONLY:
    display(template_circuit(ising(*pairwise_evidence(TOY_READS))[0], p=1)[0].draw())   # the toy puzzle, p = 1

┌───┐                                                        »
q_0: ┤ H ├─■──────────────────■────────────────■──────────────────»
     ├───┤ │ZZ(0.5*theta[0])  │                │                  »
q_1: ┤ H ├─■──────────────────┼────────────────┼──────────────────»
     ├───┤                    │ZZ(1*theta[0])  │                  »
q_2: ┤ H ├────────────────────■────────────────┼──────────────────»
     ├───┤                                     │ZZ((-1)*theta[0]) »
q_3: ┤ H ├─────────────────────────────────────■──────────────────»
     ├───┤                                                        »
q_4: ┤ H ├────────────────────────────────────────────────────────»
     └───┘                                                        »
«                                          ┌────────────────┐                  »
«q_0: ─────────────────────■───────────────┤ Ry(2*theta[1]) ├──────────────────»
«                          │               └────────────────┘┌────────────────┐»
«q_1: ─■───────────────────┼────────────────■────────────────┤ Ry(2*theta[1]) ├»
«      │ZZ((-2)*theta[0])  │                │                └────────────────┘»
«q_2: ─■───────────────────┼────────────────┼─────────────────■────────────────»
«                          │                │ZZ(1*theta[0])   │ZZ(2*theta[0])  »
«q_3: ─────────────────────┼────────────────■─────────────────■────────────────»
«                          │ZZ(1*theta[0])                                     »
«q_4: ─────────────────────■───────────────────────────────────────────────────»
«                                                                              »
«                                                             
«q_0: ────────────────────────────────────────────────────────
«                                                             
«q_1: ────────────────────────────────────────────────────────
«                         ┌────────────────┐                  
«q_2: ─■──────────────────┤ Ry(2*theta[1]) ├──────────────────
«      │                  └────────────────┘┌────────────────┐
«q_3: ─┼───────────────────■────────────────┤ Ry(2*theta[1]) ├
«      │ZZ((-1)*theta[0])  │ZZ(2*theta[0])  ├────────────────┤
«q_4: ─■───────────────────■────────────────┤ Ry(2*theta[1]) ├
«                                           └────────────────┘

## 3 · Trainer socket: choose the angles

**Gets** `J`, `offset`, the depth `p`, the circuit's socket name, and `ctx`, a dictionary of helpers:

| `ctx[...]` | What it is |
|---|---|
| `sample(angles, shots)` | runs the chosen circuit on the chosen backend and returns `shots` answers (rows of 0/1) |
| `train_shots`, `train_iters` | the budget from `Config` for shot-based training |
| `E` | exact energies of all answers when the puzzle is small (≤ 24 SNPs), else `None`. Fine for experiments, but a real device would not have them |
| `gen`, `cache` | the puzzle settings, and a dictionary shared across puzzles (e.g. to train once and reuse) |

**Returns** `(angles, shots used[, extra columns])`. The optional dictionary becomes extra columns in the results.

⚠ **Rule A:** return **every** shot you ran. The engine adds them to `tts99`, so a trainer that burns 10,000 shots cannot look cheap.

The example tries a 4 × 4 grid of one shared (γ, β), spends 64 shots on each point, and keeps the best average cost: 16 × 64 = 1,024 training shots.

In [4]:
@register("trainer", "template_grid")
def template_trainer(J, offset, p, builder, ctx):
    best, used = (np.inf, None), 0
    for g in np.linspace(0.2, 1.0, 4):
        for b in np.linspace(-0.6, 0.6, 4):
            angles = np.r_[[g] * p, [b] * p]
            energy = costs(ctx["sample"](angles, 64), J, offset).mean()
            used += 64
            if energy < best[0]:
                best = (energy, angles)
    return best[1], used, {"train_note": "4x4 grid"}

## 4 · Post-processing socket: improve the shots

**Gets** the measured `samples` (one row per shot, SNP 1 first), `J` and `offset`.

**Returns** `(samples, extra cost evaluations)`.

⚠ **Rule B:** return **one row per shot**. You may replace a row with a better answer, but never drop rows: the success rate is "optimal rows ÷ all rows". Count every cost you compute (e.g. N per shot for a one-flip polish). The engine adds them to the bill (`evals` and `tts99`).

In [5]:
@register("postprocess", "template_identity")
def template_postprocess(samples, J, offset):
    return samples, 0

## 5 · Backend socket: where the circuit runs

**Gets** `job`, `shots` and a `seed`. `job` is a dictionary with:
* `circuit`: the circuit with angles already filled in, no measurements;
* `J`, `angles`, `builder` (the circuit's socket name) and `E` (exact energies or `None`).

**Returns** `(samples, probabilities)`: `samples` has `shots` rows of 0/1 with **SNP 1 first** (Qiskit count keys are reversed: use `decode_key`). `probabilities` is the exact distribution over all $2^N$ answers if you have it, else `None`.

The example simply forwards to the built-in Aer statevector. A noisy simulator (FakeManila) or a real IBM device goes here.

In [6]:
@register("backend", "template_sv")
def template_backend(job, shots, seed):
    return REGISTRY["backend"]["aer_sv"](job, shots, seed)

## 6 · Race with your plug-ins

`Config` picks one entry per socket by name. ⚠ **Rule D:** always compare inside **one** `run_bench` call (or with identical `ns`, `instances`, `seed` and `gen`), so every method sees the same puzzles and the same budget.

Here: a 6-SNP puzzle, depth p = 1, 128 shots, all five template entries.

In [7]:
if not PLUGINS_ONLY:
    cfg = Config(methods=("qaoa",), shots=128, ps=(1,), evidence="template_pairwise", qaoa="template_ry",
                 trainer="template_grid", postprocess="template_identity", backend="template_sv")
    mine = pd.DataFrame(run_bench([6], 1, cfg))
    display(mine[["n", "method", "evidence", "builder", "trainer", "backend", "train_shots", "train_note",
                  "P_budget", "tts99", "score"]])

N=6: 6.0s  P(optimum within budget): qaoa-p1=0.00


,n,method,evidence,builder,trainer,backend,train_shots,train_note,P_budget,tts99,score
0,6,qaoa-p1,template_pairwise,template_ry,template_grid,template_sv,1024,4x4 grid,0.0,inf,0.674419


**Self-check.** The row must show all five template entries, the 1,024 training shots, and a `tts99` that includes them. A `fail` column would mean a plug-in raised an error (the engine records it instead of stopping).

In [8]:
if not PLUGINS_ONLY:
    r = mine.iloc[0]
    assert "fail" not in mine or mine["fail"].isna().all(), mine.get("fail")
    assert (r.method, r.evidence, r.builder, r.trainer, r.backend) == \
        ("qaoa-p1", "template_pairwise", "template_ry", "template_grid", "template_sv")
    assert r.train_shots == 16 * 64 and r.train_note == "4x4 grid"
    assert r.tts99 >= r.train_shots
    print("all five sockets plugged in; training shots counted in tts99")

all five sockets plugged in; training shots counted in tts99


### 6.1 Only a circuit? The default trainer

With `trainer="auto"`, a new circuit is trained by **COBYLA on shots**: `train_iters` steps (30 · p when 0) of `train_shots` shots each, all counted. It runs on Aer statevector up to 26 SNPs.

In [9]:
if not PLUGINS_ONLY:
    cfg = Config(methods=("qaoa",), shots=128, qaoa="template_ry", train_shots=32, train_iters=8)
    (r,) = run_instance(make_instance(6, seed=3), cfg, log=lambda *_: None)
    assert "fail" not in r
    assert (r["trainer"], r["backend"], r["train"]) == ("cobyla", "aer_sv", "cobyla")
    assert r["train_shots"] == 32 * r["train_evals"] > 0
    print(f"auto trainer = {r['trainer']} on {r['backend']}: {r['train_evals']} evaluations x 32 shots = {r['train_shots']} training shots")

auto trainer = cobyla on aer_sv: 8 evaluations x 32 shots = 256 training shots


## 7 · Compare with the baseline

The fair comparison: the same puzzles, the same budget, the baseline next to your entry. Random guessing and local search stay in as the floor and the classical bar. Here: 3 puzzles each of 8 and 10 SNPs with realistic read errors.

Read the table per size: higher `P_budget` and `score` are better; lower `tts99` (shots to be 99% sure, training included) and `switch_err` are better.

Expect the RY example to **lose**: it only shows the wiring. Its `tts99` also carries the 30 × 256 = 7,680 COBYLA training shots (rule A), while the baseline's closed-form training costs 0 shots.

In [10]:
if not PLUGINS_ONLY:
    gen = dict(error=0.05, missing=0.1)
    base = run_bench([8, 10], 3, Config(methods=("random", "local", "qaoa"), gen=gen), log=lambda *_: None)
    mine = run_bench([8, 10], 3, Config(methods=("qaoa",), qaoa="template_ry", gen=gen), log=lambda *_: None)
    df = pd.concat([pd.DataFrame(base), pd.DataFrame(mine).assign(method="template_ry-p1")])
    display(summarize(df))

P_budget      tts99   score  switch_err  wall_s
n  method                                                         
8  local             1.0000    79.7263  1.0000         0.0  0.0377
   qaoa-p1           1.0000    33.6672  1.0000         0.0  0.0008
   random            1.0000   587.1562  1.0000         0.0  0.0001
   template_ry-p1    0.9818  9011.5438  1.0000         0.0  0.1259
10 local             1.0000   120.3860  1.0000         0.0  0.0384
   qaoa-p1           1.0000    79.0722  1.0000         0.0  0.0008
   random            0.9818  2355.5438  1.0000         0.0  0.0001
   template_ry-p1    0.0000        inf  0.8868         1.0  0.1282

## 8 · Merge day: everyone's plug-ins in one race

The merge notebook (`bench/all_together.ipynb`) loads the engine and then each of our notebooks with `PLUGINS_ONLY = True`. Only the `@register` cells do work, so loading takes seconds. Loading the engine again inside each notebook keeps the plug-ins registered so far.

```python
QGD_QUIET = PLUGINS_ONLY = True
%run qgd_bench.ipynb
%run alexandra.ipynb      # registers e.g. alexandra_weighted
%run carla.ipynb          # carla_ws, carla_cvar
%run alex.ipynb           # alex_fake, alex_polish
import pandas as pd

cfg = Config(ps=(1, 2, 3), gen=dict(error=0.05, missing=0.1),
             evidence="alexandra_weighted", qaoa="carla_ws", trainer="carla_cvar",
             backend="alex_fake", postprocess="alex_polish")
summarize(pd.DataFrame(run_bench(["toy", 8, 10, 12, 14, 16], 5, cfg)))
```

The names are placeholders: use the ones you register.